# Dense Passage Retriever
In this notebook, we use pretrained dense passage retriever to retrieve relevant chunks of context.

## Acknowledgement

The following implementation is modified from the article **Dense Passage Retrieval (DPR): Smart Search for Intelligent Applications** by Jay Kim. Available at
https://medium.com/@bravekjh/dense-passage-retrieval-dpr-smart-search-for-intelligent-applications-82575d6dfad3.
Retrieved on November 2025

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Upload Data downloaded from the dataset

In [ ]:
!cp -r drive/MyDrive/NLP/* /content/
!pip install -r requirements.txt
!mkdir data
!unzip "/content/longbench.zip" -d "/content/data/"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 kB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 6.4 MB/s eta 0:00:00
  Created wheel for wavedrom: filename=wavedrom-2.0.3.post3-py2.py3-none-any.whl size=30142 sha256=ad8cbaed2e6a9d4f92f9312bb0201c02d1876f6a880ad96290b182ffda33ecee
  

In [ ]:
import os
from datasets import load_from_disk
from tqdm import tqdm
import json
import torch

def get_dataset(data_path, name):
  full_path = os.path.join(data_path, name)
  data = load_from_disk(full_path)

  return list(data)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Get Tokenizers and Decoders

In [ ]:
from transformers import DPRConfig, DPRContextEncoderTokenizer, \
DPRContextEncoderTokenizerFast, DPRQuestionEncoderTokenizer, \
DPRQuestionEncoderTokenizerFast, DPRReaderTokenizer, \
DPRReaderTokenizerFast,\
DPRQuestionEncoder, DPRContextEncoder
import torch
import torch.nn.functional as F


c_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base", device_map = device)
c_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base", device_map = device)
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base", device_map = device)
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base", device_map = device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# Use the "LM" tokenizer instead of DPR tokenizer, to ensure equal length for LM tokenizer
from transformers import AutoTokenizer
llm_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct", device_map = device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
def passage_chunk(context, llm_tokenizer, chunk_size, overlap):
  tokenized_passage = llm_tokenizer(context, return_tensors = "pt", padding=True, truncation=True)
  texts_ret = []
  tokens_ret = []
  index = 0
  while True:
    token_ret = tokenized_passage['input_ids'][0][index:index + chunk_size]
    if token_ret.shape[0] < chunk_size:
      #right padding
      token_ret = F.pad(token_ret, (0, chunk_size - token_ret.shape[0],), "constant", token_ret[-1]).cpu()
    document = llm_tokenizer.decode(token_ret, skip_special_tokens = True)
    tokens_ret.append(token_ret.tolist())
    texts_ret.append(document)
    index += chunk_size - overlap
    if index >= len(tokenized_passage['input_ids'][0]) - overlap:
      break

  return texts_ret, tokens_ret


In [ ]:
def retrieve_relevant_context(query, passages, c_tokenizer, c_encoder, q_tokenizer, q_encoder, top_k):
  q_embed = None
  c_inputs = None
  c_embed = None

  # Encode Context
  with torch.no_grad():
    c_inputs = c_tokenizer(passages, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    c_embed = c_encoder(**c_inputs).pooler_output

  # Encode Query
  with torch.no_grad():
      q_inputs = q_tokenizer(query, return_tensors="pt", truncation = True, max_length=512).to(device)
      q_embed = q_encoder(**q_inputs).pooler_output

  c_normalized = F.normalize(c_embed)
  q_normalized = F.normalize(q_embed)

  # Cosine similarity
  search = c_normalized @ torch.transpose(q_normalized, 0, 1)
  vals, indices = torch.sort(torch.squeeze(search), 0, descending = True)
  return indices[:top_k], vals[:top_k]

In [ ]:
# Require output path
def extract_context_for_dataset(dataset_name, data_path, output_path):
  print(f"Start {dataset_name}")
  passages = get_dataset(data_path, dataset_name)
  total = len(passages)
  now = 0

  full_path = os.path.join(output_path, dataset_name + "_extracted.jsonl")
  with open(full_path, "w+") as f:
    f.write('')

  all_extra_contexts = []
  for task in passages:
    text_passages, tokenized_passages = passage_chunk(task["context"], llm_tokenizer, 128, 32)
    if now % 40 == 0:
      print(f"Completed {now} tasks")
      with open(full_path, "a+") as f:
        for e in all_extra_contexts[now-10:now]:
          json.dump(e, f, ensure_ascii=False)
          f.write('\n')
    now += 1

    query = task["input"]
    relevant_context_idx, _ = retrieve_relevant_context(query, text_passages, c_tokenizer, c_encoder, q_tokenizer, q_encoder, 16)

    extra = dict()
    extra["context"] = [text_passages[idx] for idx in relevant_context_idx]
    extra["tokenized_context"] = [tokenized_passages[idx] for idx in relevant_context_idx]
    all_extra_contexts.append(extra)

  print(f"Completed {now} tasks for {dataset_name}")

  with open(full_path, "w+") as f:
      for e in all_extra_contexts:
        json.dump(e, f, ensure_ascii=False)
        f.write('\n')

## Retrieved Context as Text and Tokens from all the datasets

In [ ]:
for dataset in ['lcc', 'multi_news']:
  try:
    extract_context_for_dataset(dataset, "data/longbench", "drive/MyDrive/NLP/extracted")
  except Exception as e:
    print(e)
    print(f"Skip {dataset}")
    continue

Start lcc
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks
Completed 240 tasks
Completed 280 tasks
Completed 320 tasks
Completed 360 tasks
Completed 400 tasks
Completed 440 tasks
Completed 480 tasks
Completed 500 tasks for lcc
Start multi_news
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks for multi_news


In [ ]:
for dataset in ['trec', 'passage_count', 'samsum','passage_retrieval_en','repobench-p','triviaqa', '2wikimqa','lcc', 'musique', 'qmsum', 'hotpotqa', 'multifieldqa_en', 'narrativeqa', 'gov_report', 'multi_news', 'qasper']:
  try:
    extract_context_for_dataset(dataset, "data/longbench", "drive/MyDrive/NLP/extracted")
  except Exception as e:
    print(e)
    print(f"Skip {dataset}")
    continue

Start trec
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks for trec
Start passage_count
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks for passage_count
Start samsum
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks for samsum
Start passage_retrieval_en
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks for passage_retrieval_en
Start repobench-p
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
Completed 160 tasks
Completed 200 tasks
Completed 240 tasks
Completed 280 tasks
Completed 320 tasks
Completed 360 tasks
Completed 400 tasks
Completed 440 tasks
Completed 480 tasks
Completed 500 tasks for repobench-p
Start triviaqa
Completed 0 tasks
Completed 40 tasks
Completed 80 tasks
Completed 120 tasks
C

In [ ]:
passages = get_dataset("/content/data/longbench/", "2wikimqa")

  5%|▌         | 10/200 [00:00<00:00, 8839.42it/s]


### Testing

In [ ]:
print(passages)

[{'input': 'Where was the wife of Francis I Rákóczi born?', 'context': 'Passage 1:\nWaldrada of Lotharingia\nWaldrada was the mistress, and later the wife, of Lothair II of Lotharingia.\n\nBiography\nWaldrada\'s family origin is uncertain. The prolific 19th-century French writer Baron Ernouf suggested that Waldrada was of noble Gallo-Roman descent, sister of Thietgaud, the bishop of Trier, and niece of Gunther, archbishop of Cologne. However, these suggestions are not supported by any evidence, and more recent studies have instead suggested she was of relatively undistinguished social origins, though still from an aristocratic milieu.\nThe Vita Sancti Deicoli states that Waldrada was related to Eberhard II, Count of Nordgau (included Strasbourg) and the family of Etichonids, though this is a late 10th-century source and so may not be entirely reliable on this question.In 855 the Carolingian king Lothar II married Teutberga, a Carolingian aristocrat and the daughter of Bosonid Boso the 

In [ ]:
# # Do not run this if already have passage
# save_path = os.path.join(save_dir, "example_passages.jsonl")
# with open(save_path, 'w+') as f:
#   for task in passages:
#     json.dump(task, f, ensure_ascii=False)
#     f.write('\n')

In [ ]:
ex = []
print(save_path)
with open(save_path, 'r') as f:
  for line in f:
    ex.append(json.loads(line))

print(ex[8]["input"])
print(len(ex))

drive/MyDrive/NLP_checkpoint/example_passages.jsonl
Where did Helena Carroll's father study?
11


In [ ]:
# @title
from transformers import DPRConfig, DPRContextEncoderTokenizer, \
DPRContextEncoderTokenizerFast, DPRQuestionEncoderTokenizer, \
DPRQuestionEncoderTokenizerFast, DPRReaderTokenizer, \
DPRReaderTokenizerFast,\
DPRQuestionEncoder, DPRContextEncoder, BertModel
import torch
import torch.nn.functional as F

# https://medium.com/@bravekjh/dense-passage-retrieval-dpr-smart-search-for-intelligent-applications-82575d6dfad3

# Sample passages (documents)
passages = [
    "The capital of France is Paris.",
    "Python is a popular programming language for data science.",
    "The Eiffel Tower is located in Paris.",
    "Dense Passage Retrieval is a method for semantic search."
]


dpr = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
bert = BertModel.from_pretrained("bert-base-uncased")

dpr.base_model.load_state_dict(bert.state_dict(), strict=False)

c_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
c_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

c_encoder.base_model.load_state_dict(bert.state_dict(), strict=False)
q_encoder.base_model.load_state_dict(bert.state_dict(), strict=False)

with torch.no_grad():
  c_inputs = c_tokenizer(passages, return_tensors="pt", padding=True, truncation=True)
  print(c_inputs)
  c_embed = c_encoder(**c_inputs).pooler_output
  print(c_embed.shape)

print(c_embed[:, :10])

# Encode query
query = "Where is the Eiffel Tower?"
with torch.no_grad():
    q_inputs = q_tokenizer(query, return_tensors="pt")
    q_embed = q_encoder(**q_inputs).pooler_output

# Search
c_normalized = F.normalize(c_embed)
q_normalized = F.normalize(q_embed)

search = c_normalized @ torch.transpose(q_normalized, 0, 1)
vals, indices = torch.sort(torch.squeeze(search), 0, descending = True)
print(indices)

# # Display results
print(f"\n🔍 Query: {query}\n")
print("Top matching passages:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {passages[idx]} (Score: {vals[i]:.4f})")
    # print(f"tokens: {c_normalized[idx,:10]}")

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRCon

{'input_ids': tensor([[  101,  1996,  3007,  1997,  2605,  2003,  3000,  1012,   102,     0,
             0,     0],
        [  101, 18750,  2003,  1037,  2759,  4730,  2653,  2005,  2951,  2671,
          1012,   102],
        [  101,  1996,  1041, 13355,  2884,  3578,  2003,  2284,  1999,  3000,
          1012,   102],
        [  101,  9742,  6019, 26384,  2003,  1037,  4118,  2005, 21641,  3945,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([4, 768])
tensor([[ 0.2962,  0.7626,  0.2001, -0.6284, -0.7100,  2.9313, -1.3370, -0.7092,
         -1.7770, -1.9321],
        [-0.0466,  0.5930,  0.0888

In [ ]:
!pip show transformers

Name: transformers
Version: 4.57.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


In [ ]:
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
model = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
input_ids = tokenizer("Hello, is my dog cute ?", return_tensors="pt")["input_ids"]
embeddings = model(input_ids).pooler_output

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
for task in ex:
  print(len(task["context"]))
  chunks = chunk_by_words(task["context"], 15)
  print(chunks[0])

28375
Passage 1: Waldrada of Lotharingia Waldrada was the mistress, and later the wife, of Lothair
28933
Passage 1: Jim Ramel Kjellgren Jim Love Ramel Kjellgren, (born 18 July 1987) is a
26698
Passage 1: Jason Moore (director) Jason Moore (born October 22, 1970) is an American director
49532
Passage 1: Betty Hall Beatrice Perin Barker Hall (March 18, 1921 – April 26, 2018)
27602
Passage 1: Henry, Lord Paulet Lord Henry Paulet (1602–1672) was an English courtier who sat
27381
Passage 1: Edward Watson, Viscount Sondes Edward Watson, Viscount Sondes (3 July 1686 – 20
29347
Passage 1: Henry de Bohun Sir Henry de Bohun (died 23 June 1314) was an
46111
Passage 1: Bill Dundee William Cruickshanks (born 24 October 1943) is a retired Scottish-born Australian
23636
Passage 1: Donnie Elbert Donnie Elbert (May 25, 1936 – January 26, 1989) was an
32482
Passage 1: Dana Blankstein Dana Blankstein-Cohen (born March 3, 1981) is the executive director of
62298
Passage 1: The Secret Invasion The Secret I

In [ ]:
for task in ex:
  print(len(task["context"]))
  passage_chunk(task["context"])

28375
torch.Size([1, 6546])
passage 1 : waldrada of lotharingia waldrada was the mistress, and later the wife, of lothair ii of lotharingia. biography waldrada's family origin is uncertain. the prolific 19th - century french writer baron ernouf suggested that waldrada was of noble gallo - roman descent, sister of thietgaud, the bishop of trier, and niece of gunther, archbishop of cologne. however, these suggestions are not supported by any
28933
torch.Size([1, 6504])
passage 1 : jim ramel kjellgren jim love ramel kjellgren, ( born 18 july 1987 ) is a swedish actor. he is the son of lotta ramel and johan h : son kjellgren and the grandchild of povel ramel. he is perhaps best known as the character jonte in the svt series eva & adam, he reprised the role in the film eva & adam – fyra fodelsedagar och ett fia
26698
torch.Size([1, 5835])
passage 1 : jason moore ( director ) jason moore ( born october 22, 1970 ) is an american director of film, theatre and television. life and career jason 

In [ ]:
def retrieve_relevant_context(query, passages, c_tokenizer, c_encoder, q_tokenizer, q_encoder, top_k):
  with torch.no_grad():
    c_inputs = c_tokenizer(passages, return_tensors="pt", padding=True, truncation=True)
    c_embed = c_encoder(**c_inputs).pooler_output

  # Encode query
  with torch.no_grad():
      q_inputs = q_tokenizer(query, return_tensors="pt")
      q_embed = q_encoder(**q_inputs).pooler_output

  # q_embed = torch.tensor([[0.1,0.2,0.3,0.4], [0.2,0.4,0.6,0.8], [-0.5,-0.5,0.5,0.5]], dtype = torch.float)
  # c_embed = torch.tensor([[1,1,1,1]], dtype = torch.float)

  # Search
  c_normalized = F.normalize(c_embed)
  print(c_normalized[:10])
  q_normalized = F.normalize(q_embed)
  print(q_normalized[:10])

  search = c_normalized @ torch.transpose(q_normalized, 0, 1)
  vals, indices = torch.sort(torch.squeeze(search), 0, descending = True)
  return indices[:top_k], vals[:top_k]


In [ ]:
query = "Where is the Eiffel Tower?"
passages = [
    "The capital of France is Paris.",
    "Python is a popular programming language for data science.",
    "The Eiffel Tower is located in Paris.",
    "Dense Passage Retrieval is a method for semantic search."
]
top_ids, vals_ids = retrieve_relevant_context(query, passages, c_tokenizer, c_encoder, q_tokenizer, q_encoder, 4)
# # Display results

print("Top matching passages:")
for i, idx in enumerate(top_ids):
    print(f"{i+1}. {passages[idx]} (Score: {vals_ids[i]:.4f})")
    # print(f"tokens: {c_normalized[idx,:10]}")

tensor([[ 0.0120, -0.0177, -0.0116,  ..., -0.0501,  0.0553,  0.0078],
        [ 0.0362, -0.0127, -0.0369,  ..., -0.0344, -0.0217,  0.0318],
        [ 0.0364,  0.0062,  0.0584,  ..., -0.0465,  0.0603, -0.0034],
        [-0.0139, -0.0149,  0.0332,  ..., -0.0421, -0.0365,  0.0024]])
tensor([[ 1.1402e-02,  5.4069e-03,  6.6695e-02,  1.6052e-02,  3.4052e-02,
         -2.3977e-02,  1.2388e-02, -1.0595e-02, -3.4077e-03, -1.9022e-02,
         -2.7809e-02, -1.0340e-02, -2.0991e-02, -2.0433e-03, -1.3856e-02,
         -2.0368e-02, -6.7668e-03, -6.6872e-03,  1.8380e-03, -3.5884e-04,
          3.8923e-03, -9.8392e-03, -2.6169e-02, -2.6227e-02,  2.2673e-02,
         -9.5562e-03, -9.0052e-03,  2.2942e-02, -3.7275e-03,  1.8579e-02,
          2.7222e-02, -5.6791e-03, -1.6833e-02, -6.3091e-02,  5.9978e-04,
          1.8496e-02,  1.4329e-02, -9.0430e-03, -5.1779e-03, -4.1215e-02,
          5.4305e-03,  1.9006e-02,  3.5178e-02,  1.8049e-02, -9.9476e-03,
          1.1525e-02, -2.7030e-02,  1.4305e-02, -1.82